# Time-Optimal CZ Gate Optimization

This notebook demonstrates time-optimal quantum control using the new `TimeOptimalController` and `TimeOptimalTrainer` classes.

**Objective**: Optimize both the gate time AND the detuning pulse to achieve a high-fidelity CZ gate.

**Key Features**:
- Rabi frequency is held constant at maximum (time-optimal by definition)
- Neural network predicts optimal gate time
- Second network generates detuning pulse conditioned on predicted time
- Joint optimization via dual optimizers

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# Add parent directory to path
sys.path.insert(0, os.path.dirname(os.getcwd()))

from qneural.neural import TimeOptimalController, TimeOptimalTrainer
from qneural.core.gates import czphi_gate
from qneural.core.metrics import unitary_fidelity

print("✓ Imports successful")
print(f"PyTorch version: {torch.__version__}")

## 1. Configuration

Set up the time-optimal controller with bounds between 5/rabi and 12/rabi.

In [ ]:
# Physical parameters
rabi_max = 25.13  # MHz (2π × 4 MHz)
target_angle = torch.tensor([np.pi])  # CZ gate

# Time bounds in units of 1/rabi_max
# Converting to seconds: time_seconds = time_normalized / rabi_max
time_bounds_normalized = (5.0, 12.0)  # in units of 1/rabi_max
time_bounds_seconds = (5.0 / rabi_max, 12.0 / rabi_max)

print("Configuration:")
print(f"  Rabi max: {rabi_max:.2f} MHz")
print(f"  Target: CZ gate (angle = π)")
print(f"  Time bounds (normalized): {time_bounds_normalized}")
print(f"  Time bounds (seconds): [{time_bounds_seconds[0]:.4f}, {time_bounds_seconds[1]:.4f}]")
print(f"  Expected optimal: ~7.6/rabi = {7.6/rabi_max:.4f} seconds")

## 2. Create Time-Optimal Controller

The controller has two networks:
1. **Time Predictor**: angle → normalized_time [0,1]
2. **Control Generator**: (angle, time) → detuning values (normalized [0,1], scaled to physical range)

In [ ]:
# Create controller with archival hyperparameters
controller = TimeOptimalController(
    time_bounds=time_bounds_normalized,
    rabi_max=rabi_max,
    detuning_range=(-2*rabi_max, 2*rabi_max),  # ±2Ω_max
    n_time_steps=301,  # High resolution for accuracy
    time_hidden_layers=3,
    time_hidden_units=45,
    control_hidden_layers=10,
    control_hidden_units=300,
    time_output_activation='sigmoid',
    weight_scale_time=1.8,
    weight_scale_control=1.55
)

# Check parameter counts
param_counts = controller.count_parameters()
print("Controller Architecture:")
print(f"  Time predictor: {param_counts['time_predictor']:,} parameters")
print(f"  Control generator: {param_counts['control_generator']:,} parameters")
print(f"  Total: {param_counts['total']:,} parameters")

# Test initial prediction
initial_time, initial_detuning_norm = controller(target_angle)
initial_detuning_phys = controller.scale_detuning(initial_detuning_norm)
print(f"\nInitial prediction:")
print(f"  Gate time: {initial_time.item():.4f} seconds ({initial_time.item()*rabi_max:.2f}/rabi)")
print(f"  Detuning (normalized): [{initial_detuning_norm.min():.3f}, {initial_detuning_norm.max():.3f}]")
print(f"  Detuning (physical): [{initial_detuning_phys.min():.2f}, {initial_detuning_phys.max():.2f}] MHz")

## 3. Setup Trainer

Create trainer with dual optimizers (different learning rates for time and control networks).

In [ ]:
# Training hyperparameters (from archival cphase_optim.ipynb)
TIME_WEIGHT = 1e-4  # Weight for time penalty in loss
TIME_LR = 1e-5      # Learning rate for time network
CONTROL_LR = 1e-4   # Learning rate for control network
EPOCHS = 500

trainer = TimeOptimalTrainer(
    controller=controller,
    nqubits=2,
    time_weight=TIME_WEIGHT,
    time_lr=TIME_LR,
    control_lr=CONTROL_LR
)

print("Trainer Configuration:")
print(f"  Time weight: {TIME_WEIGHT}")
print(f"  Time optimizer LR: {TIME_LR}")
print(f"  Control optimizer LR: {CONTROL_LR}")
print(f"  Epochs: {EPOCHS}")

## 4. Train

Optimize for single angle (CZ gate). The loss is:
```
loss = infidelity + time_weight × gate_time
```

This encourages the network to:
1. Minimize infidelity (achieve high-fidelity gate)
2. Minimize gate time (find time-optimal solution)

In [ ]:
print("Training time-optimal CZ gate...\n")

history = trainer.train(
    angles=target_angle,
    epochs=EPOCHS,
    print_every=50
)

final_loss = history['loss'][-1]
final_infidelity = history['infidelity'][-1]
final_time = history['mean_gate_time'][-1]

print(f"\n{'='*60}")
print("Training Complete!")
print(f"{'='*60}")
print(f"Final Loss: {final_loss:.6f}")
print(f"Final Infidelity: {final_infidelity:.6f}")
print(f"Final Fidelity: {(1-final_infidelity)*100:.2f}%")
print(f"Final Gate Time: {final_time:.4f} seconds ({final_time*rabi_max:.2f}/rabi)")
print(f"{'='*60}")

## 5. Visualize Training Progress

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Loss
axes[0, 0].plot(history['epoch'], history['loss'], 'b-', linewidth=2)
axes[0, 0].set_xlabel('Epoch', fontsize=12)
axes[0, 0].set_ylabel('Total Loss', fontsize=12)
axes[0, 0].set_title('Training Loss', fontsize=14, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# Infidelity
axes[0, 1].plot(history['epoch'], history['infidelity'], 'r-', linewidth=2)
axes[0, 1].set_xlabel('Epoch', fontsize=12)
axes[0, 1].set_ylabel('Infidelity', fontsize=12)
axes[0, 1].set_title('Gate Infidelity', fontsize=14, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_yscale('log')

# Gate Time
axes[1, 0].plot(history['epoch'], history['mean_gate_time'], 'g-', linewidth=2)
axes[1, 0].axhline(y=7.62/rabi_max, color='orange', linestyle='--', 
                   label='Expected optimal (~7.6/rabi)', linewidth=2)
axes[1, 0].set_xlabel('Epoch', fontsize=12)
axes[1, 0].set_ylabel('Gate Time (seconds)', fontsize=12)
axes[1, 0].set_title('Predicted Gate Time', fontsize=14, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].legend(fontsize=11)

# Fidelity
fidelities = [(1 - inf) * 100 for inf in history['infidelity']]
axes[1, 1].plot(history['epoch'], fidelities, 'purple', linewidth=2)
axes[1, 1].axhline(y=99, color='red', linestyle='--', linewidth=2, label='99% target')
axes[1, 1].set_xlabel('Epoch', fontsize=12)
axes[1, 1].set_ylabel('Fidelity (%)', fontsize=12)
axes[1, 1].set_title('Gate Fidelity', fontsize=14, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].legend(fontsize=11)
axes[1, 1].set_ylim([0, 105])

plt.tight_layout()
plt.show()

print(f"\nConvergence Summary:")
print(f"  Initial gate time: {history['mean_gate_time'][0]:.4f} s")
print(f"  Final gate time: {final_time:.4f} s")
print(f"  Time change: {((final_time - history['mean_gate_time'][0]) / history['mean_gate_time'][0] * 100):+.1f}%")
print(f"  Infidelity reduction: {((history['infidelity'][0] - final_infidelity) / history['infidelity'][0] * 100):.1f}%")

## 6. Visualize Optimized Pulses

In [ ]:
# Get final optimized pulses
controller.eval()
with torch.no_grad():
    final_gate_time, final_detuning_norm = controller(target_angle)
    
    # Scale detuning to physical range
    final_detuning = controller.scale_detuning(final_detuning_norm)
    
    # Create time array
    n_steps = controller.n_time_steps
    times = np.linspace(0, final_gate_time.item(), n_steps)
    
    # Rabi is constant
    rabi_vals = np.ones(n_steps) * rabi_max
    
    # Detuning values
    detuning_vals = final_detuning[0, :, 0].numpy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Rabi pulse (constant)
ax1.plot(times, rabi_vals, 'b-', linewidth=2.5)
ax1.fill_between(times, 0, rabi_vals, alpha=0.2, color='blue')
ax1.set_xlabel('Time (s)', fontsize=12)
ax1.set_ylabel('Rabi Frequency (MHz)', fontsize=12)
ax1.set_title('Rabi Pulse (Fixed at Ω_max)', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.set_ylim([0, rabi_max * 1.1])

# Detuning pulse (learned)
ax2.plot(times, detuning_vals, 'r-', linewidth=2.5)
ax2.fill_between(times, 0, detuning_vals, alpha=0.2, color='red')
ax2.axhline(y=0, color='k', linestyle='--', linewidth=1, alpha=0.5)
ax2.set_xlabel('Time (s)', fontsize=12)
ax2.set_ylabel('Detuning (MHz)', fontsize=12)
ax2.set_title('Detuning Pulse (NN Optimized)', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.suptitle(f'Time-Optimal CZ Gate Pulses (Fidelity: {(1-final_infidelity)*100:.2f}%, Time: {final_time:.4f}s)', 
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\nPulse Statistics:")
print(f"  Gate time: {final_time:.4f} seconds ({final_time*rabi_max:.2f}/rabi)")
print(f"  Rabi: constant {rabi_max:.2f} MHz")
print(f"  Detuning range: [{detuning_vals.min():.2f}, {detuning_vals.max():.2f}] MHz")
print(f"  Detuning mean: {detuning_vals.mean():.2f} MHz")

## 7. Verify Final Unitary

In [ ]:
# Evolve with optimized pulses to get final unitary
controller.eval()
with torch.no_grad():
    gate_time, detuning_norm = controller(target_angle)
    
    # Get pulse functions
    rabi_fn = controller.get_rabi_pulse_fn(gate_time)
    detuning_fn = controller.get_detuning_pulse_fn(detuning_norm, gate_time)
    
    # Evolve
    final_U = trainer._evolve(rabi_fn, detuning_fn, gate_time.item(), apply_corrections=True)

# Target CZ gate
target_U = czphi_gate(np.pi)

print("Achieved Unitary (diagonal elements):")
for i, val in enumerate(torch.diag(final_U)):
    phase = np.angle(val.item()) / np.pi
    mag = abs(val.item())
    print(f"  |{i:02b}⟩: {val.real:+.4f} {val.imag:+.4f}i  →  |val|={mag:.4f}, phase={phase:+.3f}π")

print("\nTarget CZ Unitary (diagonal):")
for i in range(4):
    val = 1.0 if i < 3 else -1.0
    phase = 0.0 if i < 3 else 1.0
    print(f"  |{i:02b}⟩: {val:+.4f}         →  |val|=1.0000, phase={phase:+.3f}π")

# Compute fidelity
fidelity = unitary_fidelity(final_U, target_U, dim=2, nqubits=2)
print(f"\n{'='*60}")
print(f"Final Gate Fidelity: {fidelity*100:.4f}%")
print(f"{'='*60}")

## 8. Save Model

Save the trained controller for later use or further optimization.

In [ ]:
# Save checkpoint
checkpoint_path = 'time_optimal_cz_model.pt'
trainer.save_checkpoint(checkpoint_path)

print(f"✓ Model saved to: {checkpoint_path}")
print(f"\nTo load and use:")
print(f"  checkpoint = torch.load('{checkpoint_path}')")
print(f"  controller.time_predictor.load_state_dict(checkpoint['time_network_state_dict'])")
print(f"  controller.control_generator.load_state_dict(checkpoint['control_network_state_dict'])")

## Summary

This notebook demonstrated:

1. **Time-Optimal Architecture**: Two chained networks - time predictor + control generator
2. **Dual Optimizers**: Different learning rates for time (1e-5) vs control (1e-4) networks
3. **Joint Optimization**: Loss = infidelity + λ × time encourages both fidelity and speed
4. **Results**: Converged to high-fidelity CZ gate with optimized gate time

**Key Parameters** (from archival cphase_optim.ipynb):
- Time bounds: 5/rabi to 12/rabi
- Time weight: 1e-4
- Network: 3 layers × 45 units (time), 10 layers × 300 units (control)
- Solver: RK4 with 301 time steps